# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a guide for loading, exploring, and processing the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
[https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json](https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

# Access dataset description and name
print(f"Dataset Name: {metadata.name if hasattr(metadata, 'name') else 'N/A'}")
print(f"Description: {metadata.description if hasattr(metadata, 'description') else 'N/A'}")
print(f"Published: {metadata.datePublished if hasattr(metadata, 'datePublished') else 'N/A'}")
print(f"Identifier: {metadata.identifier if hasattr(metadata, 'identifier') else 'N/A'}")

## 2. Data Overview
Review available record sets and their fields. All entities are referenced by their `@id` fields.

In [ ]:
# List all record sets and their IDs
record_sets = dataset.record_sets
print("Available Record Sets:")
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']}, name: {rs.get('name', 'N/A')}")

# List fields for each record set
for rs in record_sets:
    print(f"\nFields in RecordSet {rs['@id']}: {rs.get('name', 'N/A')}")
    for field in rs.get('field', []):
        if isinstance(field, dict):
            print(f"  - Field @id: {field.get('@id', 'N/A')} (name: {field.get('name', 'N/A')})")
        else:
            print(f"  - Field @id: {field}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.
Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data for each record set by @id
dataframes = {}

# Get list of record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
print("Record Set @ids:", record_set_ids)

for record_set_id in record_set_ids:
    print(f"\nLoading records for RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns for RecordSet @id {record_set_id}: {df.columns.tolist()}")
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping data.
All fields and columns are referenced by their `@id`.

In [ ]:
# Example: Use first record set and a numeric field (if available)
if len(record_set_ids) > 0:
    rs_id = record_set_ids[0]
    df = dataframes[rs_id]
    # Find a numeric-like field
    numeric_fields = [col for col in df.columns if 'age' in col.lower() or 'height' in col.lower()]
    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"Using numeric field '{numeric_field}' for filtering.")
        
        # Filter records: age/height > threshold
        threshold = 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Grouping: if field with 'oligomenorrhea' or similar
        group_field = None
        for col in df.columns:
            if 'oligomenorrhea' in col.lower():
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            print(grouped_df.head())
    else:
        print("No numeric field found for EDA in RecordSet.")
else:
    print("No record sets available.")

## 5. Visualization
Visualize data distributions or relationships using fields referenced by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot numeric field distribution
if len(record_set_ids) > 0 and numeric_fields:
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field} (@id: '{numeric_field}')")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If group_field was found, plot mean and std by group
    if group_field:
        group_stats = df.groupby(group_field)[numeric_field].agg(['mean', 'std'])
        group_stats.plot.bar()
        plt.title(f"{numeric_field} grouped by {group_field} (@id)")
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
This notebook demonstrated loading and exploring a FAIR² dataset using the `mlcroissant` library.
We reviewed record sets and fields by their `@id`, performed basic filtering and normalization, grouped by key attributes, and visualized data distributions.
Due to the small sample size (N=3), interpret findings cautiously; this structured exploration framework enables scalable FAIR-compliant analysis for more extensive datasets.